In [ ]:
from pathlib import Path
import os
import numpy as np
import arviz as az
import pickle
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import h5py  # Make sure h5py initialises properly before pandas ruins it

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, plot_post_prior_comparison, plot_prior_multipost

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
for c in countries[:10]:
    print(c)
    analyses = {d.name: Path(d.path) for d in os.scandir(job_path / c) if d.is_dir()}
    
    try:
        no_mob_path = analyses["no_mob"]
    
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analyses.items()}
        targs = load_targets(no_mob_path)
        plot_multianalysis_fit(c, targs, spaghs)
        
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
        plot_prior_multipost(idatas, 4, priors)
    except:
        print(f"couldn't plot country: {c}")